# 🔍 Basic Retrieval Methods for RAG Systems

## What You'll Learn

Once documents are chunked, you need to retrieve the most relevant ones. This notebook covers **5 essential retrieval methods** that form the foundation of RAG systems.

**You'll master:**
1. 🟢 **Similarity Search** - The default (start here!)
2. 🟡 **MMR** - Diverse results
3. 🟠 **Metadata Filtering** - Targeted search
4. 🔵 **Reranking** - Precision boost
5. 🟣 **Hybrid Search** - Best of both worlds

**Time to complete:** ~15 minutes  
**Difficulty:** Beginner-friendly

---

## How Retrieval Works

```
User Query → Embedding → Vector Search → Top-K Chunks → LLM Context
```

Different methods optimize different parts of this pipeline!

## Setup

We'll use the chunks you created in the chunking notebook.

In [ ]:
# Import essentials
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from retrieval_playground.utils import constants
from retrieval_playground.utils.model_manager import model_manager
from retrieval_playground.src.pre_retrieval.chunking_manager import ChunkingStrategy

# Use recursive chunking (from 1A notebook)
strategy = ChunkingStrategy.RECURSIVE_CHARACTER

# Connect to vector database
qdrant_client = QdrantClient(url=constants.QDRANT_URL, api_key=constants.QDRANT_KEY)
embeddings = model_manager.get_embeddings()

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=strategy.value,
    embedding=embeddings
)

print("✅ Connected to vector database!")
print(f"Using collection: {strategy.value}")

In [ ]:
# Sample query for testing
query = "What is chain-of-thought prompting?"
TOP_K = 3  # Number of results to retrieve

print(f"Test query: {query}")
print(f"Retrieving top {TOP_K} results")

## 🟢 Method 1: Similarity Search (START HERE)

### Why Start Here?
- ✅ **Simple**: One line of code
- ✅ **Fast**: Milliseconds response time
- ✅ **Effective**: Works well for 80% of use cases
- ✅ **Default**: Industry standard

### How It Works
Finds chunks most semantically similar to your query using cosine similarity:

```
Query Embedding → Compare with all chunks → Return Top-K most similar
```

### When to Use
- 📄 Default choice for RAG
- 🚀 Need fast results
- 💡 General Q&A systems

**→ Use this unless you have a specific reason not to!**

In [ ]:
# Basic similarity search with scores
results = vector_store.similarity_search_with_relevance_scores(query, k=TOP_K)

print("🟢 Similarity Search Results:\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.3f}")
    print(f"   Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:100]}...\n")

from qdrant_client.models import Filter, FieldCondition, MatchText
from qdrant_client.http import models as rest

# Create index for metadata filtering (required for efficient filtering)
print("📝 Creating metadata index for filtering...")
try:
    qdrant_client.create_payload_index(
        collection_name=strategy.value,
        field_name="metadata.source",
        field_schema=rest.PayloadSchemaType.TEXT
    )
    print("✅ Index created!\n")
except Exception as e:
    print(f"ℹ️  Index may already exist: {e}\n")

# Example: Search only in chain-of-thought paper
target_file = "cot_paper.pdf"

filtered_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": TOP_K,
        "filter": Filter(
            must=[
                FieldCondition(
                    key="metadata.source",
                    match=MatchText(text=target_file)
                )
            ]
        )
    }
)

filtered_docs = filtered_retriever.invoke(query)

print(f"🟠 Metadata Filtering Results:")
print(f"Searching only in: {target_file}\n")

for i, doc in enumerate(filtered_docs, 1):
    print(f"{i}. Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:80]}...\n")

In [ ]:
# Try different thresholds
THRESHOLD = 0.7  # Only scores above 0.7

retriever = vector_store.as_retriever(
    search_kwargs={"k": TOP_K, "score_threshold": THRESHOLD}
)

filtered_results = retriever.invoke(query)
print(f"Found {len(filtered_results)} results above threshold {THRESHOLD}\n")

for i, doc in enumerate(filtered_results, 1):
    print(f"{i}. {doc.page_content[:80]}...")

try:
    from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever
    
    # Initialize hybrid retriever
    hybrid = HybridRetriever(
        strategy=strategy,
        qdrant_client=qdrant_client
    )
    
    # Search with hybrid method
    hybrid_results = hybrid.search(query, k=TOP_K)
    
    print("🟣 Hybrid Search Results:\n")
    for i, doc in enumerate(hybrid_results, 1):
        rrf_score = doc.metadata.get('rrf_score', 0)
        print(f"{i}. RRF Score: {rrf_score:.4f}")
        print(f"   Preview: {doc.page_content[:80]}...\n")
    
    hybrid.close()
    
except ImportError as e:
    print("⚠️  Hybrid search requires rank-bm25 package")
    print("Install with: pip install rank-bm25")
    print(f"\nError: {e}")

In [ ]:
# MMR search
query_embedding = embeddings.embed_query(query)
mmr_results = vector_store.max_marginal_relevance_search_with_score_by_vector(
    embedding=query_embedding, 
    k=TOP_K
)

print("🟡 MMR Results (Diverse):\n")
for i, (doc, score) in enumerate(mmr_results, 1):
    print(f"{i}. MMR Score: {score:.3f}")
    print(f"   Preview: {doc.page_content[:80]}...\n")

In [ ]:
# Compare: Similarity vs MMR
print("📊 Comparison:")
print("\nSimilarity (may be redundant):")
for i, (doc, _) in enumerate(results[:2], 1):
    print(f"{i}. {doc.page_content[:60]}...")

print("\nMMR (more diverse):")
for i, (doc, _) in enumerate(mmr_results[:2], 1):
    print(f"{i}. {doc.page_content[:60]}...")

print("\n💡 Notice: MMR may return different chunks!")

## 🟠 Method 3: Metadata Filtering

### What It Is
Filter search by document properties (source, date, type, etc.):

```
Query: "What is BERT?"
Filter: Only NLP papers from 2023
Result: BERT info from NLP papers only
```

### When to Use
- 🏢 **Multi-domain systems**: Search only medical, legal, or technical docs
- 📅 **Time-based**: Only recent documents
- 🔐 **Permissions**: User-specific access control
- 📂 **Source filtering**: Specific files or categories

**Trade-off:** May miss relevant content from filtered sources

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchText

# Example: Search only in chain-of-thought paper
target_file = "cot_paper.pdf"

filtered_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": TOP_K,
        "filter": Filter(
            must=[
                FieldCondition(
                    key="metadata.source",
                    match=MatchText(text=target_file)
                )
            ]
        )
    }
)

filtered_docs = filtered_retriever.invoke(query)

print(f"🟠 Metadata Filtering Results:")
print(f"Searching only in: {target_file}\n")

for i, doc in enumerate(filtered_docs, 1):
    print(f"{i}. Source: {doc.metadata.get('source', 'N/A')}")
    print(f"   Preview: {doc.page_content[:80]}...\n")

## 🔵 Method 4: Reranking

### The Two-Stage Approach

**Stage 1: Fast Retrieval** (get candidates)
```
Similarity search → Top 50 candidates
```

**Stage 2: Precise Reranking** (pick best)
```
Cross-encoder reranks 50 → Top 3 best
```

### How It Works
Uses a more powerful (slower) model to re-score candidates:

- **Similarity search**: Fast but approximate (~1ms)
- **Cross-encoder**: Slow but precise (~100ms)

### When to Use
- 🎯 **Precision critical**: Medical, legal, financial
- 💰 **Small result sets**: Top 3-5 for LLM context
- 🔍 **Complex queries**: Nuanced semantic matching

**Trade-off:** Higher latency, better quality

In [ ]:
from retrieval_playground.src.mid_retrieval.reranking import Reranker

# Initialize reranker
reranker = Reranker(
    strategy=strategy,
    qdrant_client=qdrant_client,
    top_n=TOP_K
)

# Rerank results
reranked = reranker.retrieve(query)

print("🔵 Reranking Results:\n")
for i, doc in enumerate(reranked, 1):
    score = doc.metadata.get('rerank_score', 0)
    print(f"{i}. Rerank Score: {score:.3f}")
    print(f"   Preview: {doc.page_content[:80]}...\n")

reranker.close_qdrant_client()

## 🟣 Method 5: Hybrid Search (BM25 + Dense)

### Best of Both Worlds

**Dense Search (Embeddings)**:
- Query: "What is BERT?"
- Finds: "transformer model", "language understanding"
- ✅ Great for semantic meaning
- ❌ May miss exact keywords

**BM25 (Keyword)**:
- Query: "What is BERT?"
- Finds: "BERT", exact matches
- ✅ Great for exact terms
- ❌ May miss paraphrases

**Hybrid = Dense + BM25**:
- ✅ Exact matches + Semantic understanding

### When to Use
- 🔍 **Technical queries**: Acronyms, codes, specific terms
- 📚 **Mixed content**: Some exact, some semantic
- 💪 **Production systems**: Maximum recall

**Trade-off:** More complex, requires BM25 index

In [ ]:
from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever

# Initialize hybrid retriever
hybrid = HybridRetriever(
    strategy=strategy,
    qdrant_client=qdrant_client
)

# Search with hybrid method
hybrid_results = hybrid.search(query, k=TOP_K)

print("🟣 Hybrid Search Results:\n")
for i, doc in enumerate(hybrid_results, 1):
    rrf_score = doc.metadata.get('rrf_score', 0)
    print(f"{i}. RRF Score: {rrf_score:.4f}")
    print(f"   Preview: {doc.page_content[:80]}...\n")

hybrid.close()

### 🔬 Experiment: Compare All Methods

Let's compare Dense vs BM25 vs Hybrid:

In [ ]:
# Reinitialize for comparison
hybrid = HybridRetriever(strategy=strategy, qdrant_client=qdrant_client)

comparison = hybrid.compare_methods(query, k=3)

print("📊 Method Comparison:\n")
print("=" * 80)

print("\n🔤 BM25 (Keyword Matching):")
for i, doc in enumerate(comparison['bm25'][:2], 1):
    print(f"{i}. {doc.page_content[:60]}...")

print("\n🧠 Dense (Semantic):")
for i, doc in enumerate(comparison['dense'][:2], 1):
    print(f"{i}. {doc.page_content[:60]}...")

print("\n🚀 Hybrid (BM25 + Dense):")
for i, doc in enumerate(comparison['hybrid'][:2], 1):
    print(f"{i}. {doc.page_content[:60]}...")

print("\n💡 Hybrid combines strengths of both!")

hybrid.close()

## 🎯 Decision Guide

### Quick Decision Tree

```
START
    |
    ├─ First time? Simple use case?
    │   └─ YES → 🟢 Similarity Search
    |
    ├─ Getting duplicate/similar results?
    │   └─ YES → 🟡 Add MMR
    |
    ├─ Need to filter by source/domain?
    │   └─ YES → 🟠 Metadata Filtering
    |
    ├─ Need maximum precision (top 3)?
    │   └─ YES → 🔵 Reranking
    |
    └─ Technical queries with exact terms?
        └─ YES → 🟣 Hybrid Search
```

### Use Case Examples

| Your Situation | Recommended Method | Why |
|----------------|--------------------|-----------|
| First RAG project | 🟢 Similarity | Simple, works well |
| Repetitive content | 🟡 MMR | Adds diversity |
| Multi-domain KB | 🟠 Metadata Filter | Targeted search |
| High-stakes queries | 🔵 Reranking | Maximum precision |
| Technical docs | 🟣 Hybrid | Exact + semantic |

### Common Combinations

**Production RAG**:
```python
Hybrid Search → Reranking → Top 3 to LLM
```

**Multi-domain System**:
```python
Metadata Filter → Similarity Search → MMR
```

**Simple Chatbot**:
```python
Similarity Search (with score threshold)
```

## 📚 Quick Reference

### Copy-Paste Code

**Similarity Search:**
```python
results = vector_store.similarity_search_with_relevance_scores(query, k=3)
```

**MMR:**
```python
embedding = embeddings.embed_query(query)
results = vector_store.max_marginal_relevance_search_with_score_by_vector(
    embedding=embedding, k=3
)
```

**Metadata Filter:**
```python
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": Filter(
            must=[FieldCondition(key="metadata.source", match=MatchText(text="file.pdf"))]
        )
    }
)
```

**Reranking:**
```python
from retrieval_playground.src.mid_retrieval.reranking import Reranker
reranker = Reranker(strategy=strategy, qdrant_client=client, top_n=3)
results = reranker.retrieve(query)
```

**Hybrid:**
```python
from retrieval_playground.src.mid_retrieval.hybrid_search import HybridRetriever
hybrid = HybridRetriever(strategy=strategy, qdrant_client=client)
results = hybrid.search(query, k=3)
```

## 🎉 Congratulations!

You've learned 5 essential retrieval methods:

✅ 🟢 **Similarity Search** - Your default choice  
✅ 🟡 **MMR** - When you need diversity  
✅ 🟠 **Metadata Filtering** - When you need targeting  
✅ 🔵 **Reranking** - When precision matters most  
✅ 🟣 **Hybrid Search** - When you need both exact + semantic  

### Next Steps

1. **Try with your queries** - Test different methods!
2. **Learn advanced methods** - Check `2B_Advanced_Mid_Retrieval_Methods.ipynb`
3. **Combine with post-retrieval** - See `3_Post_Retrieval.ipynb`

### Pro Tips

- 🚀 **Start simple**: Similarity search first
- 📊 **Measure impact**: Compare before/after metrics
- 🔄 **Iterate**: Add complexity only when needed
- 💰 **Balance cost vs quality**: Reranking is expensive but effective

**Remember:** 80% of use cases work great with just Similarity Search! 🎯